# 通过openai调用云端大模型
演示如何远程调用千问大模型。调用方式可能随模型迭代发生改变，最新版见[官方开发文档](https://help.aliyun.com/zh/model-studio/qwen-api-reference)。

In [ ]:
from openai import OpenAI

class LLMClient():
    def __init__(self, my_key):
        workspace_id = 'your_workspace_id'
        self.client = OpenAI(api_key=my_key, base_url=f'https://{workspace_id}.cn-beijing.maas.aliyuncs.com/compatible-mode/v1')
        
    def _process (self, prompt, model="qwen3-max"):
        messages = [{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content":prompt}]
        completion = self.client.chat.completions.create(model=model, messages=messages)
        resp = completion.to_dict()['choices'][0]['message']['content']
        return resp

    def answer(self, prompt):
        return self._process(prompt)
        
# 测试代码
llm = LLMClient("your_own_api_key")
resp = llm.answer("最小的鸟类是什么？")
print(resp)

print(resp)的输出

世界上最小的鸟类是**蜂鸟**中的**蜜蜂蜂鸟**（学名：Mellisuga helenae），也被称为**古巴蜜蜂蜂鸟**。

- **体长**：约5至6厘米（包括喙和尾羽）。
- **体重**：仅约1.6至2克，比一枚硬币还轻。
- **分布**：仅分布于古巴及其邻近的青年岛（Isla de la Juventud）。
- **特征**：雄鸟羽毛呈鲜艳的红绿色，雌鸟则偏灰绿色。它们飞行灵活，能悬停、倒飞，翅膀每秒可拍动80次以上。
- **食性**：主要以花蜜为食，也吃小型昆虫和蜘蛛以补充蛋白质。

蜜蜂蜂鸟不仅是体型最小的鸟，也是新陈代谢率最高的脊椎动物之一，每天需采食数百朵花才能维持能量需求。

# 基于transformers部署本地大模型

In [2]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id="Qwen/Qwen3-0.6B", local_dir="./local_llms/Qwen3-0.6B", endpoint="https://hf-mirror.com")

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

'/Users/weiqijie/Desktop/教材/python-book-main/notebooks/local_llms/Qwen3-0.6B'

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer  
        
class LocalLLMClient(LLMClient):
    def __init__(self, model_path):
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForCausalLM.from_pretrained(model_path, dtype="auto", device_map="auto")

    def _process(self, prompt):
        messages = [{"role": "user", "content": prompt}]
        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)
        generated_ids = self.model.generate(**model_inputs, max_new_tokens=10000)
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
        resp = self.tokenizer.decode(output_ids, skip_special_tokens=True)
        return resp
model_path = 'local_llms/Qwen3-0.6B'
llm = LocalLLMClient(model_path)
resp = llm.answer("最小的鸟类是什么？") 
print(resp)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

最小的鸟类是**鹦鹉**。它们通常体型较小，喙短而小，适应于栖息在各种环境中，如树上、岩石或灌木丛中。
